In [ ]:
import math
import re
import datasets
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.nn import functional as F


# 1. DOWNLOAD E TOKENIZAÇÃO POR ESPAÇOS/PALAVRAS

print('Baixando amostra da Wikipédia em Português...')
dataset = load_dataset('wikimedia/wikipedia', '20231101.pt', split='train[:3000]')
raw_text = '\n\n'.join(dataset['text'])


def clean_wiki_text(t):
  t = re.sub(r'http\S+', '', t)
  t = re.sub(r'==+.*?==+', '', t)
  t = re.sub(r'[\x91\x92\x93\x94\x96\x97\x9c\xa0\xad]', '', t)
  t = re.sub(r'[ \t]+', ' ', t)
  t = re.sub(r'\n+', '\n', t)
  return t


text = clean_wiki_text(raw_text)

# Tokenizador simples por espaços (separa palavras e pontuações grudadas)
# Usamos regex para isolar pontuações como vírgulas e pontos em "tokens" próprios
def tokenize(text):
  # Adiciona espaços ao redor de pontuações para que elas virem tokens separados
  text = re.sub(r'([,.\?!;:])', r' \1 ', text)
  return text.split()


tokens = tokenize(text)

# Cria o vocabulário de palavras únicas
vocab = sorted(list(set(tokens)))
vocab_size = len(vocab)

stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for i, word in enumerate(vocab)}

encode = lambda s: [stoi[w] for w in tokenize(s) if w in stoi]
decode = lambda l: ' '.join([itos[i] for i in l])

# Transforma em Tensores PyTorch
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f'Total de palavras no texto: {len(data)}')
print(f'Tamanho do vocabulário (palavras únicas): {vocab_size}')



# 2. ARQUITETURA DO TRANSFORMER (ADAPTADA)
# Mudamos o block_size para 16 ou 32
block_size = 32  # 32 palavras de contexto


class Head(nn.Module):

  def __init__(self, head_size, n_embd, block_size, dropout):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer(
        'tril', torch.tril(torch.ones(block_size, block_size))
    )
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape
    k = self.key(x)
    q = self.query(x)
    wei = q @ k.transpose(-2, -1) * (C ** -0.5)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    v = self.value(x)
    return wei @ v


class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
    super().__init__()
    self.heads = nn.ModuleList([
        Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)
    ])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    return self.dropout(self.proj(out))


class FeedForward(nn.Module):

  def __init__(self, n_embd, dropout):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.GELU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):

  def __init__(self, n_embd, n_head, block_size, dropout):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
    self.ffwd = FeedForward(n_embd, dropout)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


class GPTWordModel(nn.Module):

  def __init__(
      self, vocab_size, n_embd, n_head, n_layer, block_size, dropout
  ):
    super().__init__()
    self.block_size = block_size
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        *[
            Block(n_embd, n_head, block_size, dropout)
            for _ in range(n_layer)
        ]
    )
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(
        torch.arange(T, device=idx.device)
    )
    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B * T, C)
      targets = targets.view(B * T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens, temperature=0.8):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.block_size :]
      logits, _ = self(idx_cond)
      logits = logits[:, -1, :] / temperature
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx



# 3. TREINAMENTO E GERAÇÃO
def treinar_gpt(
    train_data,
    val_data,
    vocab_size,
    block_size,
    max_iters=3000,
    batch_size=32,
    n_embd=128,
    n_head=4,
    n_layer=4,
    dropout=0.1,
    lr=1e-3,
):
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(f'Treinando no dispositivo: {device.upper()}')

  def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

  model = GPTWordModel(
      vocab_size, n_embd, n_head, n_layer, block_size, dropout
  ).to(device)
  optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

  eval_interval = 500
  for iter in range(max_iters):
    if iter % eval_interval == 0:
      model.eval()
      with torch.no_grad():
        xb, yb = get_batch('val')
        _, val_loss = model(xb, yb)
        print(f'Passo {iter:4d} | Perda de Validação: {val_loss.item():.4f}')
      model.train()

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

  print('\nTreinamento concluído!')
  return model, device


def gerar_texto_gpt(
    model,
    device,
    stoi,
    itos,
    prompt,
    block_size=32,
    max_words=50,
    temp=0.8,
):
  model.eval()
  words = tokenize(prompt)
  encoded = [stoi.get(w, 0) for w in words]
  if not encoded:
    encoded = [0]

  context = torch.tensor([encoded], dtype=torch.long, device=device)

  with torch.no_grad():
    for _ in range(max_words):
      idx_cond = context[:, -block_size:]
      logits, _ = model(idx_cond)
      logits = logits[:, -1, :] / temp
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      context = torch.cat((context, idx_next), dim=1)

  generated_tokens = context[0].tolist()
  # Junta as palavras respeitando os espaços de pontuação
  text_out = ''
  for i, idx in enumerate(generated_tokens):
    word = itos.get(idx, '')
    if word in [',', '.', '!', '?', ';', ':']:
      text_out += word  # Pontuação gruda na palavra anterior
    else:
      text_out += ' ' + word
  return text_out.strip()



# 4. EXECUTANDO O TREINO E TESTES
modelo_treinado, dispositivo = treinar_gpt(
    train_data=train_data,
    val_data=val_data,
    vocab_size=vocab_size,
    block_size=block_size,
    max_iters=3000,
)

print('\n--- Resultado por Palavras ---')
print(
    gerar_texto_gpt(
        modelo_treinado,
        dispositivo,
        stoi,
        itos,
        prompt='Napoleão Bonaparte foi',
        max_words=40,
        temp=0.7,
    )
)

Baixando amostra da Wikipédia em Português...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.pt/train-00000-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  426MB            

20231101.pt/train-00000-of-00006.parquet: downloading bytes:           |  0.00B            

20231101.pt/train-00001-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  212MB            

20231101.pt/train-00001-of-00006.parquet: downloading bytes:           |  0.00B            

... (6 outputs left out)

20231101.pt/train-00005-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  292MB            

20231101.pt/train-00005-of-00006.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1112246 [00:00<?, ? examples/s]

Total de palavras no texto: 6855047
Tamanho do vocabulário (palavras únicas): 279477
Treinando no dispositivo: CUDA
Passo    0 | Perda de Validação: 12.7436
Passo  500 | Perda de Validação: 7.4400
Passo 1000 | Perda de Validação: 7.0238
Passo 1500 | Perda de Validação: 6.5919
Passo 2000 | Perda de Validação: 6.4589
Passo 2500 | Perda de Validação: 6.7378

Treinamento concluído!

--- Resultado por Palavras ---
Napoleão Bonaparte foi de 0, a capital e a União Internacional de São Paulo, o país foi fundamental, composta por um período de área de uma população de maior, em Pyongyang e uma grande dimensão da população. O


In [ ]:
print('\n--- Resultado por Palavras ---')
print(
    gerar_texto_gpt(
        modelo_treinado,
        dispositivo,
        stoi,
        itos,
        prompt='3',
        max_words=40,
        temp=0.7,
    )

NameError: name 'x' is not defined